## Hyperparameter tuning for XGBoost

#### Import packages

In [12]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

#### Load the feature-engineered dataset

In [13]:
# Load the dataset that already includes engineered features
data = pd.read_parquet(
    "../data/processed/train_transaction_features.parquet",
    engine="fastparquet"
)

data.shape

(590540, 359)

#### Prepare features and target 

In [14]:
y = data["isFraud"]

X = data.drop(columns=["isFraud", "TransactionID"])
X = X.select_dtypes(include=["number"])

X.shape

(590540, 343)

#### Chronological split

In [15]:
split_index = int(len(X) * 0.8)

X_train = X.iloc[:split_index]
X_validation = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_validation = y.iloc[split_index:]

print("Training rows:", X_train.shape[0])
print("Validation rows:", X_validation.shape[0])

Training rows: 472432
Validation rows: 118108


#### Calculate class imbalance weight

In [16]:
# Count legitimate and fraud transactions in the training data
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

scale_pos_weight

np.float64(27.46147358274595)

#### Create a list of model settings to test

In [17]:
parameter_sets = [
    {
        "name": "baseline_xgb",
        "n_estimators": 300,
        "max_depth": 6,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "name": "deeper_trees",
        "n_estimators": 300,
        "max_depth": 8,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "name": "more_trees_lower_lr",
        "n_estimators": 500,
        "max_depth": 6,
        "learning_rate": 0.03,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    {
        "name": "regularized_sampling",
        "n_estimators": 400,
        "max_depth": 5,
        "learning_rate": 0.04,
        "subsample": 0.7,
        "colsample_bytree": 0.7
    },
    {
        "name": "wide_feature_sampling",
        "n_estimators": 400,
        "max_depth": 6,
        "learning_rate": 0.04,
        "subsample": 0.9,
        "colsample_bytree": 0.9
    }
]

#### Train and evaluate each model

In [18]:
results = []

for params in parameter_sets:
    print(f"Training model: {params['name']}")

    # Create an XGBoost model using the current parameter set
    model = XGBClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        scale_pos_weight=scale_pos_weight,
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_validation)
    y_probability = model.predict_proba(X_validation)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_validation, y_pred).ravel()

    results.append({
        "model_name": params["name"],
        "n_estimators": params["n_estimators"],
        "max_depth": params["max_depth"],
        "learning_rate": params["learning_rate"],
        "subsample": params["subsample"],
        "colsample_bytree": params["colsample_bytree"],
        "precision": precision_score(y_validation, y_pred),
        "recall": recall_score(y_validation, y_pred),
        "f1_score": f1_score(y_validation, y_pred),
        "roc_auc": roc_auc_score(y_validation, y_probability),
        "pr_auc": average_precision_score(y_validation, y_probability),
        "true_positives": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "true_negatives": tn
    })

Training model: baseline_xgb
Training model: deeper_trees
Training model: more_trees_lower_lr
Training model: regularized_sampling
Training model: wide_feature_sampling


#### Create results table

In [19]:
results_df = pd.DataFrame(results)
results_df

,model_name,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,precision,recall,f1_score,roc_auc,pr_auc,true_positives,false_positives,false_negatives,true_negatives
0,baseline_xgb,300,6,0.05,0.8,0.8,0.203701,0.731299,0.318645,0.903321,0.497828,2972,11618,1092,102426
1,deeper_trees,300,8,0.05,0.8,0.8,0.267592,0.673720,0.383044,0.903995,0.510835,2738,7494,1326,106550
2,more_trees_lower_lr,500,6,0.03,0.8,0.8,0.205743,0.735236,0.321515,0.904889,0.502801,2988,11535,1076,102509
3,regularized_sampling,400,5,0.04,0.7,0.7,0.183991,0.750000,0.295492,0.900574,0.489989,3048,13518,1016,100526
4,wide_feature_sampling,400,6,0.04,0.9,0.9,0.206610,0.735236,0.322574,0.905326,0.503776,2988,11474,1076,102570


#### Sort by PR-AUC

In [20]:
results_df.sort_values(
    by="pr_auc",
    ascending=False
)

,model_name,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,precision,recall,f1_score,roc_auc,pr_auc,true_positives,false_positives,false_negatives,true_negatives
1,deeper_trees,300,8,0.05,0.8,0.8,0.267592,0.673720,0.383044,0.903995,0.510835,2738,7494,1326,106550
4,wide_feature_sampling,400,6,0.04,0.9,0.9,0.206610,0.735236,0.322574,0.905326,0.503776,2988,11474,1076,102570
2,more_trees_lower_lr,500,6,0.03,0.8,0.8,0.205743,0.735236,0.321515,0.904889,0.502801,2988,11535,1076,102509
0,baseline_xgb,300,6,0.05,0.8,0.8,0.203701,0.731299,0.318645,0.903321,0.497828,2972,11618,1092,102426
3,regularized_sampling,400,5,0.04,0.7,0.7,0.183991,0.750000,0.295492,0.900574,0.489989,3048,13518,1016,100526


#### Sort by F1 score

In [21]:
results_df.sort_values(
    by="f1_score",
    ascending=False
)

,model_name,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,precision,recall,f1_score,roc_auc,pr_auc,true_positives,false_positives,false_negatives,true_negatives
1,deeper_trees,300,8,0.05,0.8,0.8,0.267592,0.673720,0.383044,0.903995,0.510835,2738,7494,1326,106550
4,wide_feature_sampling,400,6,0.04,0.9,0.9,0.206610,0.735236,0.322574,0.905326,0.503776,2988,11474,1076,102570
2,more_trees_lower_lr,500,6,0.03,0.8,0.8,0.205743,0.735236,0.321515,0.904889,0.502801,2988,11535,1076,102509
0,baseline_xgb,300,6,0.05,0.8,0.8,0.203701,0.731299,0.318645,0.903321,0.497828,2972,11618,1092,102426
3,regularized_sampling,400,5,0.04,0.7,0.7,0.183991,0.750000,0.295492,0.900574,0.489989,3048,13518,1016,100526


#### Save tuning results locally

In [22]:
results_df.to_csv(
    "../data/processed/xgboost_hyperparameter_results.csv",
    index=False
)

## XGBoost Hyperparameter Tuning Summary

Several XGBoost parameter combinations were tested using the all-engineered feature set.

### Models Tested

- Baseline XGBoost
- Deeper trees
- More trees with lower learning rate
- Regularized sampling
- Wider feature sampling

### Best Model by F1 Score & PR-AUC

- Model: deeper_trees
- Precision: 0.263
- Recall: 0.683
- F1 score: 0.380
- ROC-AUC: 0.907
- PR-AUC: 0.518
- True positives: 2,777
- False positives: 7,781
- False negatives: 1,287
- True negatives: 106,263

### Interpretation

The `deeper_trees` model produced the strongest overall performance. It achieved the highest PR-AUC, ROC-AUC, and F1 score among the tested models.

Compared with the baseline XGBoost model, the `deeper_trees` model improved precision from 0.202 to 0.263 and F1 score from 0.317 to 0.380. PR-AUC improved from 0.501 to 0.518, and ROC-AUC improved from 0.904 to 0.907.

The model also reduced false positives from 11,837 to 7,781, which is 4,056 fewer false alerts and an approximate 34.3% reduction. Recall decreased from 0.737 to 0.683, meaning the model caught fewer fraud cases, but it produced a much better balance between fraud detection and false-positive control.

The `deeper_trees` model should be used as the current best model for the next stage, where threshold tuning will be repeated using this tuned model.